# Building heights, floors and vertical structure
**Inputs:** a study-area shapefile, OSM buildings, Google Open Buildings 2.5D Temporal heights (Earth Engine).
**Outputs:** per-building heights and floor estimates (CSV + GeoPackage), grid summaries and figures in `OUT_DIR`.

Run the cells top to bottom. Change only **Cell 1 (Settings)**.

In [ ]:
# Run once per environment, then comment out
# !pip install earthengine-api geemap osmnx geopandas matplotlib pydeck

## 1. Settings

In [ ]:
import os, time, json, glob
import numpy as np, pandas as pd, geopandas as gpd
import ee, geemap, osmnx as ox
from shapely.geometry import mapping, box
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display

PROJECT   = 'your-ee-project-id'
SHP       = r'path/to/study_area.shp'   # study-area polygon(s)
AREA_NAME = 'bengaluru'                                      # used in output file names
OUT_DIR   = r'outputs'      # all results and figures go here
CACHE_DIR = r'cache'          # re-uses downloads already made (OSM buildings, Earth Engine batches); set = OUT_DIR to start fresh
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
YEAR      = 2023

CHUNK_SIZE = 2000      # buildings per Earth Engine request (keeps requests < 10 MB)
WORKERS    = 6         # parallel requests

GROUND_FLOOR_M = 4.0             # used by the simple 'formula' model only
FLOOR_MODEL_CHOICE = 'linear'     # 'linear' (recommended), 'isotonic', 'formula', or 'auto' = lowest cross-validated error
MIN_HEIGHT_M   = 2.5

GRID_M      = 500      # cell size for the grid maps (metres)
VIEW_CENTER = None     # (lat, lon) for the 3D view; None = around the tallest building
VIEW_RADIUS = 700      # metres around VIEW_CENTER shown in 3D

FIG_DIR = os.path.join(OUT_DIR, f'figures_{AREA_NAME}')
os.makedirs(FIG_DIR, exist_ok=True)

### Chart style (shared by all figures)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch

INK, INK2, MUTED, GRID, AXIS, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
BLUE = '#2a78d6'
SEQ  = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b']   # light -> dark
NODATA = '#c3c2b7'

# Floor bands (ordinal): each band one step darker
BANDS      = [0, 1, 3, 6, 10, 15, 1000]
BAND_NAMES = ['1', '2–3', '4–6', '7–10', '11–15', '16+']
BAND_COLS  = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#104281', '#0d366b']

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Segoe UI', 'Arial', 'DejaVu Sans'],
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'text.color': INK, 'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.edgecolor': AXIS, 'axes.linewidth': 0.8, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'savefig.dpi': 200, 'savefig.bbox': 'tight',
})

def save(fig, name):
    base = os.path.join(FIG_DIR, os.path.splitext(name)[0])
    fig.savefig(base + '.png')                 # raster, 200 dpi
    fig.savefig(base + '.svg')                 # vector: text, axes and outlines stay editable
    print('saved', base + '.png / .svg')

def floor_band(floors):
    return pd.cut(floors, BANDS, labels=BAND_NAMES)

## 2. Connect to Earth Engine

In [ ]:
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('Earth Engine ready')

## 3. Study area

In [ ]:
aoi = gpd.read_file(SHP)
if aoi.crs is None:
    raise ValueError('Shapefile has no CRS (.prj file missing)')
aoi = aoi.to_crs(4326)
aoi_geom = (aoi.geometry.union_all() if hasattr(aoi.geometry, 'union_all')
            else aoi.geometry.unary_union).buffer(0)
UTM = aoi.estimate_utm_crs()                     # metric CRS for the city (works for any Indian city)
print('Study area bounds:', [round(b, 4) for b in aoi_geom.bounds], '| UTM:', UTM.to_epsg())

## 4. OSM buildings
Downloaded once, then loaded from a local cache. Tries backup servers if one is down.

In [ ]:
osm_cache = os.path.join(CACHE_DIR, f'osm_buildings_{AREA_NAME}.gpkg')
if os.path.exists(osm_cache):
    gdf = gpd.read_file(osm_cache)
    print('Loaded cached OSM buildings:', len(gdf))
else:
    ox.settings.requests_timeout = 600
    gdf = None
    for url in ['https://overpass-api.de/api',
                'https://overpass.private.coffee/api',
                'https://overpass.kumi.systems/api',
                'https://maps.mail.ru/osm/tools/overpass/api']:
        try:
            ox.settings.overpass_url = url
            print('Trying', url)
            gdf = ox.features_from_polygon(aoi_geom, tags={'building': True})
            print('  OK')
            break
        except Exception as e:
            print('  failed:', type(e).__name__); time.sleep(5)
    if gdf is None:
        raise RuntimeError('All OSM servers failed - check internet/VPN')

    gdf = gdf[gdf.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index()
    gdf = gdf[gdf.to_crs(UTM).representative_point().to_crs(4326).within(aoi_geom).values]
    gdf = gdf.rename(columns={'id': 'osmid', 'element': 'osm_type', 'element_type': 'osm_type'})
    keep = [c for c in ['osm_type', 'osmid', 'building', 'building:levels', 'name', 'height', 'geometry']
            if c in gdf.columns]
    gdf = gdf[keep].to_crs(4326)
    gdf.columns = [c.replace(':', '_') for c in gdf.columns]
    for c in ['building_levels', 'name', 'building']:
        if c not in gdf.columns:
            gdf[c] = np.nan
    gdf['bid'] = gdf['osm_type'].astype(str).str[0] + gdf['osmid'].astype(str)
    gdf['building_levels'] = pd.to_numeric(gdf['building_levels'], errors='coerce')
    gdf['name'] = gdf['name'].fillna('').astype(str)
    gdf['building'] = gdf['building'].fillna('yes').astype(str)
    gdf = gdf[['bid', 'osmid', 'osm_type', 'building', 'building_levels', 'name', 'geometry']].reset_index(drop=True)
    gdf.to_file(osm_cache, driver='GPKG')
print(len(gdf), 'buildings')

## 5. Height layer: interactive map

In [ ]:
aoi_ee = ee.Geometry(mapping(aoi_geom.simplify(0.0005)))      # small, simplified boundary
col = ee.ImageCollection('GOOGLE/Research/open-buildings-temporal/v1').filterBounds(aoi_ee)
print('Height images (all years):', col.size().getInfo())
h = col.filterDate(f'{YEAR}-01-01', f'{YEAR+1}-01-01').select('building_height').mosaic()

Map = geemap.Map()
Map.centerObject(aoi_ee, 11)
Map.addLayer(h, {'min': 0, 'max': 60, 'palette': ['white','yellow','orange','red','purple']}, f'Height {YEAR}')
Map.add_gdf(aoi, layer_name='Study area', style={'color': 'black', 'fillOpacity': 0})
display(Map)

## 6. Height per building (batched)
Resumable: if it stops, run the cell again and finished batches are skipped.

In [ ]:
# Only ID + slightly simplified outline are sent; attributes stay on your computer.
reducer = ee.Reducer.median().combine(ee.Reducer.percentile([90]), '', True)
chunk_dir = os.path.join(CACHE_DIR, f'_chunks_{AREA_NAME}')
os.makedirs(chunk_dir, exist_ok=True)

send = gdf[['bid', 'geometry']].copy()
send['geometry'] = send.geometry.simplify(0.000005)     # ~0.5 m, shrinks request size
chunks = [send.iloc[i:i + CHUNK_SIZE] for i in range(0, len(send), CHUNK_SIZE)]
print(f'{len(chunks)} batches of up to {CHUNK_SIZE} buildings')

def run_chunk(i):
    path = os.path.join(chunk_dir, f'chunk_{i:05d}.csv')
    if os.path.exists(path):                 # already done (lets you resume after a crash)
        return i, 'cached'
    fc = ee.FeatureCollection(json.loads(chunks[i].to_json()))
    st = (h.reduceRegions(collection=fc, reducer=reducer, scale=4, tileScale=4)
            .map(lambda f: f.select(['bid', 'median', 'p90'], None, False)))   # drop outlines from the reply
    for attempt in range(4):
        try:
            feats = st.getInfo()['features']
            rows = [f['properties'] for f in feats]
            pd.DataFrame(rows, columns=['bid', 'median', 'p90']).to_csv(path, index=False)
            return i, 'ok'
        except Exception as e:
            err = str(e)
            time.sleep(10 * (attempt + 1))
    return i, 'FAILED: ' + err[:120]

t0, failed = time.time(), []
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = [ex.submit(run_chunk, i) for i in range(len(chunks))]
    for n, fut in enumerate(as_completed(futures), 1):
        i, status = fut.result()
        if status.startswith('FAILED'):
            failed.append(i); print(f'batch {i}: {status}')
        if n % 20 == 0 or n == len(chunks):
            print(f'{n}/{len(chunks)} batches done  ({(time.time() - t0) / 60:.1f} min)')
if failed:
    print('Failed batches:', failed, '-> just run this section again; finished batches are skipped.')

hts = pd.concat([pd.read_csv(p) for p in glob.glob(os.path.join(chunk_dir, 'chunk_*.csv'))], ignore_index=True)
hts = hts.drop_duplicates('bid').rename(columns={'median': 'h_median', 'p90': 'h_p90'})
res = gdf.merge(hts, on='bid', how='left')
res[['h_median', 'h_p90']] = res[['h_median', 'h_p90']].fillna(0)   # no height pixels = 0
print('Buildings with heights:', hts['bid'].nunique(), 'of', len(gdf))

## 7. Quality check

In [ ]:
print(res[['h_median', 'h_p90']].describe().round(1))
print('Height = 0          :', (res['h_median'] == 0).sum())
print(f'Height < {MIN_HEIGHT_M} m      :', (res['h_median'] < MIN_HEIGHT_M).sum())
print('With OSM levels tag :', res['building_levels'].notna().sum())

res_m = res.to_crs(UTM)
res['footprint_m2'] = res_m.area.round(1)
cen = res_m.centroid.to_crs(4326)
res['lon'], res['lat'] = cen.x.round(6), cen.y.round(6)

## 8. Calibrate the floor model
Three ways of turning satellite height into floors are compared with 5-fold cross-validation on the OSM-tagged buildings:
* **formula**: ground floor + fixed storey height (the earlier method)
* **linear**: straight line fitted to the tagged buildings (robust to bad tags)
* **isotonic**: rising step curve fitted to the tagged buildings

OSM tags of 0, negative or above 60 floors are treated as missing.

In [ ]:
# OSM floor tags of 0, negative or > 60 are treated as missing
res.loc[~res['building_levels'].between(1, 60), 'building_levels'] = np.nan

cal = res[res['building_levels'].notna() & (res['h_median'] >= MIN_HEIGHT_M)].copy()
print('Tagged buildings with a usable height:', len(cal))

from sklearn.linear_model import HuberRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold

def fit_models(h, lv):
    """Three ways to turn satellite height into floors. Returns {name: predict(height) -> continuous floors}."""
    imp = (h - GROUND_FLOOR_M) / (lv - 1).where(lv > 1)                      # storey height implied by each building
    storey = float(np.nanmedian(imp[(imp > 2) & (imp < 6)])) if (lv > 1).sum() else 3.2
    hub = HuberRegressor().fit(h.values.reshape(-1, 1), lv.values)            # robust to bad tags
    iso = IsotonicRegression(out_of_bounds='clip').fit(h.values, lv.values)
    return {'formula':  lambda x, s=storey: (np.asarray(x) - GROUND_FLOOR_M) / s + 1,
            'linear':   lambda x, m=hub: m.predict(np.asarray(x).reshape(-1, 1)),
            'isotonic': lambda x, m=iso: m.predict(np.asarray(x))}, storey, hub

def to_floors(raw):
    return np.clip(np.round(raw), 1, None)

# first pass: flag tags that disagree wildly with height (likely tagging or height errors)
models, storey_formula, hub = fit_models(cal['h_median'], cal['building_levels'])
resid = to_floors(models['linear'](cal['h_median'])) - cal['building_levels']
cal['plausible'] = resid.abs() <= np.maximum(3, 0.4 * cal['building_levels'])
cal['storey_implied'] = (cal['h_median'] - GROUND_FLOOR_M) / (cal['building_levels'] - 1).where(cal['building_levels'] > 1)
good = cal[cal['plausible']]
print(f'Plausible: {len(good):,} | excluded as likely tag/height errors: {(~cal.plausible).sum():,}')

# 5-fold cross-validation of the three models on the plausible tagged buildings
cv = []
if len(good) >= 30:
    for tr, te in KFold(5, shuffle=True, random_state=0).split(good):
        m, _, _ = fit_models(good['h_median'].iloc[tr], good['building_levels'].iloc[tr])
        for name, f in m.items():
            e = to_floors(f(good['h_median'].iloc[te])) - good['building_levels'].iloc[te].values
            cv.append({'model': name, 'MAE': np.abs(e).mean(), 'within_1': (np.abs(e) <= 1).mean(), 'bias': e.mean()})
    cv = pd.DataFrame(cv).groupby('model').mean().reindex(['formula', 'linear', 'isotonic'])
    display(cv.round(3))
    FLOOR_MODEL = FLOOR_MODEL_CHOICE if FLOOR_MODEL_CHOICE != 'auto' else cv['MAE'].idxmin()
else:
    FLOOR_MODEL = 'formula'
    print('Fewer than 30 tagged buildings: using the simple formula')

models, storey_formula, hub = fit_models(good['h_median'], good['building_levels'])   # final fit on all plausible
storey_other = (1 / hub.coef_[0]) if FLOOR_MODEL == 'linear' else storey_formula        # metres per floor (for charts)
print(f'Floor model used: {FLOOR_MODEL}', '| linear fit: floors = '
      f'{hub.intercept_:.2f} + {hub.coef_[0]:.3f} × height  (≈ {1 / hub.coef_[0]:.2f} m per floor)')

def estimate_floors(height, storey=None, model=None):
    """Floors from satellite height with the chosen model; NaN where there is no usable height."""
    height = pd.Series(height)
    f = pd.Series(to_floors(models[model or FLOOR_MODEL](height.fillna(0))), index=height.index)
    return f.where(height >= MIN_HEIGHT_M)

## 9. Floors and quality flags

In [ ]:
res['floors_est']         = estimate_floors(res['h_median'])
res['floors_est_p90']     = estimate_floors(res['h_p90'])
res['floors_est_formula'] = estimate_floors(res['h_median'], model='formula')    # previous method, for comparison
res['floor_model']        = FLOOR_MODEL

h_med, lv, fe, fa = res['h_median'], res['building_levels'], res['floors_est'], res['footprint_m2']
flag_rules = {
    'no_height':        h_med == 0,
    'very_low':         (h_med > 0) & (h_med < MIN_HEIGHT_M),
    'levels_conflict':  lv.notna() & fe.notna() & ((fe - lv).abs() > np.maximum(3, 0.4 * lv)),
    'possible_complex': fa > 5000,
    'tag_check':        (res['building'] == 'house') & (h_med > 20),
}
res['qc_flags'] = ''
for name, mask in flag_rules.items():
    res.loc[mask, 'qc_flags'] += name + ';'
res['qc_flags'] = res['qc_flags'].str.rstrip(';')

use_osm = lv.notna() & ~flag_rules['levels_conflict']
res['floors_final'] = np.where(use_osm, lv, fe)
res['floor_source'] = np.where(use_osm, 'osm', np.where(fe.notna(), 'estimated', 'none'))
print(res['floor_source'].value_counts())
print({k: int(v.sum()) for k, v in flag_rules.items()})

## 10. Save results

In [ ]:
cols = ['bid', 'osmid', 'name', 'building', 'building_levels', 'h_median', 'h_p90', 'floor_model',
        'floors_est', 'floors_est_p90', 'floors_est_formula', 'floors_final', 'floor_source', 'footprint_m2', 'lat', 'lon', 'qc_flags']
out_csv  = os.path.join(OUT_DIR, f'buildings_floors_{AREA_NAME}.csv')
out_gpkg = os.path.join(OUT_DIR, f'buildings_floors_{AREA_NAME}.gpkg')   # GeoPackage: faster than GeoJSON for big cities
res[cols].to_csv(out_csv, index=False)
res[cols + ['geometry']].to_file(out_gpkg, driver='GPKG')
print('Saved:', out_csv, '\n       ', out_gpkg)

## 11. Change 2016 → 2023 (optional)
Heights and building presence for 2016–17 and 2022–23 for every building, classified as new / taller / lower / stable.
Saves `building_change_<AREA>.csv`, used by the entropy notebook. Takes about as long as section 6; resumable.

In [ ]:
EARLY, LATE = [2016, 2017], [2022, 2023]

def period_image(years, tag):
    imgs = [col.filterDate(f'{y}-01-01', f'{y+1}-01-01').select(['building_height', 'building_presence']).mosaic()
            for y in years]
    return ee.ImageCollection(imgs).mean().rename([f'h_{tag}', f'p_{tag}'])

change_img = period_image(EARLY, 'early').addBands(period_image(LATE, 'late'))
chg_dir = os.path.join(CACHE_DIR, f'_chunks_change_{AREA_NAME}')
os.makedirs(chg_dir, exist_ok=True)

def run_change_chunk(i):
    path = os.path.join(chg_dir, f'chunk_{i:05d}.csv')
    if os.path.exists(path):
        return i, 'cached'
    fc = ee.FeatureCollection(json.loads(chunks[i].to_json()))
    st = (change_img.reduceRegions(collection=fc, reducer=ee.Reducer.median(), scale=4, tileScale=4)
          .map(lambda f: f.select(['bid', 'h_early', 'p_early', 'h_late', 'p_late'], None, False)))
    for attempt in range(4):
        try:
            rows = [f['properties'] for f in st.getInfo()['features']]
            pd.DataFrame(rows, columns=['bid', 'h_early', 'p_early', 'h_late', 'p_late']).to_csv(path, index=False)
            return i, 'ok'
        except Exception as e:
            err = str(e); time.sleep(10 * (attempt + 1))
    return i, 'FAILED: ' + err[:120]

t0, failed = time.time(), []
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = [ex.submit(run_change_chunk, i) for i in range(len(chunks))]
    for n, fut in enumerate(as_completed(futures), 1):
        i, status = fut.result()
        if status.startswith('FAILED'): failed.append(i); print(f'batch {i}: {status}')
        if n % 20 == 0 or n == len(chunks): print(f'{n}/{len(chunks)} batches  ({(time.time() - t0) / 60:.1f} min)')
if failed: print('Failed batches:', failed, '-> run this cell again')

chg = pd.concat([pd.read_csv(p) for p in glob.glob(os.path.join(chg_dir, 'chunk_*.csv'))], ignore_index=True).drop_duplicates('bid')
chg = chg.merge(res[['bid', 'footprint_m2', 'lat', 'lon']], on='bid', how='left')

# ---- classify each building ----
P_ABSENT, P_PRESENT = 0.3, 0.5          # building_presence is an uncalibrated confidence: treat these as tunable
MIN_DH = 3.0                            # metres: smaller height changes are treated as noise
chg['floors_early'] = estimate_floors(chg['h_early'])
chg['floors_late']  = estimate_floors(chg['h_late'])
chg['d_height'] = chg['h_late'] - chg['h_early']
chg['d_floors'] = (chg['floors_late'] - chg['floors_early']).fillna(0)
new   = (chg['p_early'] < P_ABSENT) & (chg['p_late'] >= P_PRESENT) & (chg['h_late'] >= MIN_HEIGHT_M)
grew  = ~new & (chg['p_early'] >= P_PRESENT) & (chg['d_height'] >= MIN_DH) & (chg['d_floors'] >= 1)
lower = ~new & (chg['p_early'] >= P_PRESENT) & (chg['d_height'] <= -MIN_DH) & (chg['d_floors'] <= -1)
chg['change'] = np.select([new, grew, lower], ['new', 'taller', 'lower'], 'stable')
chg['added_floor_area_m2'] = np.where(new, chg['footprint_m2'] * chg['floors_late'].fillna(1),
                              np.where(grew, chg['footprint_m2'] * chg['d_floors'], 0))
chg.to_csv(os.path.join(OUT_DIR, f'building_change_{AREA_NAME}.csv'), index=False)
print(chg['change'].value_counts())

---
# Visualisations
Every figure is also saved as PNG in `FIG_DIR`.

### V1. Height and floor distribution

In [ ]:
res['floor_band'] = floor_band(res['floors_final'])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))

h = res.loc[res['h_median'] >= MIN_HEIGHT_M, 'h_median']
a1.hist(h, bins=np.arange(0, min(h.max(), 150) + 3, 3), color=BLUE, edgecolor=SURFACE, linewidth=1)
a1.axvline(h.median(), color=INK2, lw=1, ls='--')
a1.annotate(f'median {h.median():.1f} m', (h.median(), a1.get_ylim()[1] * 0.92),
            xytext=(6, 0), textcoords='offset points', color=INK2, fontsize=9)
a1.set(title='Building height', xlabel='Median height (m)', ylabel='Buildings')
a1.grid(axis='x', visible=False)

counts = res['floor_band'].value_counts().reindex(BAND_NAMES).fillna(0)
nodata = res['floor_band'].isna().sum()
bars = a2.bar(BAND_NAMES, counts.values, color=BAND_COLS, edgecolor=SURFACE, linewidth=2)
a2.bar_label(bars, labels=[f'{v:,.0f}\n{v / len(res):.0%}' for v in counts.values], fontsize=8.5, color=INK2, padding=2)
a2.set(title='Floors per building', xlabel='Floors', ylabel='Buildings')
a2.margins(y=0.18); a2.grid(axis='x', visible=False)
a2.text(1, -0.2, f'No usable height: {nodata:,} buildings', transform=a2.transAxes, ha='right', color=MUTED, fontsize=8.5)

fig.suptitle(f'{AREA_NAME.title()}: {len(res):,} OSM buildings', x=0.01, ha='left', fontsize=13, fontweight='bold')
fig.tight_layout(); save(fig, 'v1_height_floor_distribution.png'); plt.show()

### V2. Where the floor counts come from, and quality flags

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={'width_ratios': [1, 1.3]})

src_order = ['osm', 'estimated', 'none']
src_lab   = {'osm': 'OSM levels tag', 'estimated': 'Estimated from height', 'none': 'No estimate'}
src_col   = {'osm': '#2a78d6', 'estimated': '#86b6ef', 'none': NODATA}
sc = res['floor_source'].value_counts().reindex(src_order).fillna(0)
left = 0
for k in src_order:
    a1.barh(0, sc[k], left=left, color=src_col[k], edgecolor=SURFACE, linewidth=2, height=0.5)
    if sc[k] / sc.sum() > 0.06:
        a1.text(left + sc[k] / 2, 0, f'{sc[k] / sc.sum():.0%}', ha='center', va='center',
                color='white' if k == 'osm' else INK, fontsize=9)
    left += sc[k]
a1.set(title='Source of floor count', yticks=[], xlabel='Buildings')
a1.legend(handles=[Patch(color=src_col[k], label=f'{src_lab[k]} ({sc[k]:,.0f})') for k in src_order],
          loc='upper center', bbox_to_anchor=(0.5, -0.35), ncol=1, frameon=False, fontsize=9)
a1.grid(axis='y', visible=False)

fc = pd.Series({k: int(v.sum()) for k, v in flag_rules.items()}).sort_values()
flag_lab = {'no_height': 'No height', 'very_low': f'Height < {MIN_HEIGHT_M} m', 'levels_conflict': 'OSM levels ≠ height',
            'possible_complex': 'Footprint > 5,000 m²', 'tag_check': "'house' taller than 20 m"}
b = a2.barh([flag_lab[k] for k in fc.index], fc.values, color=BLUE, height=0.55)
a2.bar_label(b, labels=[f'{v:,}' for v in fc.values], padding=3, fontsize=9, color=INK2)
a2.set(title='Buildings flagged for checking', xlabel='Buildings'); a2.grid(axis='y', visible=False)
a2.margins(x=0.15)
a2.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))

fig.tight_layout(); save(fig, 'v2_sources_and_flags.png'); plt.show()

### V3. Calibration: satellite height vs OSM floor tags, and model comparison

In [ ]:
if len(cal):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 5), gridspec_kw={'width_ratios': [1.5, 1]})
    MAXF = 25
    ok, bad = cal[cal['plausible']].copy(), cal[~cal['plausible']]
    ok['L'] = ok['building_levels'].clip(upper=MAXF)
    rng = np.random.default_rng(0)
    alpha = float(np.clip(30 / max(len(ok), 1), 0.06, 0.7))
    a1.scatter(ok['L'] + rng.uniform(-0.28, 0.28, len(ok)), ok['h_median'], s=6, color=BLUE, alpha=alpha,
               linewidths=0, rasterized=True, label=f'Tagged buildings ({len(ok):,})', zorder=2)
    g = ok.groupby('L')['h_median'].describe(percentiles=[.25, .5, .75]); g = g[g['count'] >= 3]
    a1.vlines(g.index, g['25%'], g['75%'], color=INK, lw=2, zorder=4)
    a1.scatter(g.index, g['50%'], s=28, color=SURFACE, edgecolor=INK, linewidth=1.4, zorder=5,
               label='Median and middle 50% per floor count')
    hg = np.linspace(MIN_HEIGHT_M, np.nanpercentile(ok['h_median'], 99.5), 300)
    a1.plot(models['formula'](hg), hg, color=MUTED, lw=1.4, ls='--', zorder=3,
            label=f'Formula: {GROUND_FLOOR_M:.0f} m + {storey_formula:.2f} m per floor')
    if FLOOR_MODEL != 'formula':
        a1.plot(models[FLOOR_MODEL](hg), hg, color=INK2, lw=2.2, zorder=3, label=f'{FLOOR_MODEL.title()} fit (used)')
    a1.set_xlim(0.3, min(ok['L'].max(), MAXF) + 1); a1.set_ylim(0, hg.max() * 1.05)
    a1.set_title('Satellite height vs OSM floor count', pad=20)
    a1.text(0, 1.02, f'{len(ok):,} plausible tagged buildings  ·  {len(bad):,} excluded as likely tag/height errors',
            transform=a1.transAxes, fontsize=8.5, color=INK2, va='bottom')
    a1.set(xlabel='Floors (OSM building:levels)', ylabel='Median satellite height (m)')
    leg = a1.legend(frameon=False, fontsize=8, loc='upper left', markerscale=1.6)
    for h_ in leg.legend_handles:
        if isinstance(h_, mpl.collections.PathCollection): h_.set_alpha(1)

    if isinstance(cv, pd.DataFrame) and len(cv):
        names = list(cv.index)
        cols_ = [BLUE if n == FLOOR_MODEL else '#86b6ef' for n in names]
        bars = a2.barh(names, cv['within_1'], color=cols_, height=0.55)
        for y, n in enumerate(names):
            r = cv.loc[n]
            a2.text(r['within_1'] + 0.01, y, f"{r['within_1']:.0%}   MAE {r['MAE']:.2f}   bias {r['bias']:+.2f}",
                    va='center', fontsize=8.5, color=INK2)
        a2.set_xlim(0, 1.3); a2.set_xticks([0, .25, .5, .75, 1]); a2.invert_yaxis()
        a2.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
        a2.set_title('Cross-validated accuracy', pad=20)
        a2.text(0, 1.02, 'Share of held-out buildings within ±1 floor (darker = model used)',
                transform=a2.transAxes, fontsize=8.5, color=INK2, va='bottom')
        a2.set(xlabel='Within ±1 floor'); a2.grid(axis='y', visible=False)
    else:
        a2.set_axis_off()
    fig.tight_layout(); save(fig, 'v3_calibration.png'); plt.show()
else:
    print('No OSM-tagged buildings to plot.')

### V3b. Before / after: floor distribution with the old formula and the fitted model

In [ ]:
bins, labs = [0, 1, 2, 3, 5, 10, 1000], ['1', '2', '3', '4–5', '6–10', '11+']
old = pd.cut(res['floors_est_formula'], bins, labels=labs).value_counts(normalize=True).reindex(labs)
new = pd.cut(res['floors_est'], bins, labels=labs).value_counts(normalize=True).reindex(labs)
x = np.arange(len(labs)); w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 4.4))
b1 = ax.bar(x - w / 2, old.values, w, color='#86b6ef', label='Formula (before)')
b2 = ax.bar(x + w / 2, new.values, w, color=BLUE, label=f'{FLOOR_MODEL.title()} fit (after)')
for bars_ in (b1, b2):
    ax.bar_label(bars_, labels=[f'{v:.0%}' if v >= 0.005 else '' for v in bars_.datavalues], fontsize=8, color=INK2, padding=2)
ax.set_xticks(x, labs); ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
ax.set(title='Estimated floors per building: before and after calibration', xlabel='Floors', ylabel='Share of buildings')
ax.grid(axis='x', visible=False); ax.margins(y=0.15); ax.legend(frameon=False)
fig.tight_layout(); save(fig, 'v3b_before_after.png'); plt.show()

### V4. Map: buildings by number of floors
Large cities are drawn as points for speed.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
plot = res.to_crs(UTM)
aoi.to_crs(UTM).boundary.plot(ax=ax, color=INK2, linewidth=0.8, zorder=1)
nod = plot[plot['floor_band'].isna()]
as_points = len(plot) > 150_000

def draw(df, color, z):
    if as_points:
        ax.scatter(df.centroid.x, df.centroid.y, s=0.3, color=color, linewidths=0, zorder=z, rasterized=True)
    else:
        df.plot(ax=ax, color=color, linewidth=0, zorder=z, rasterized=len(df) > 5000)

draw(nod, NODATA, 2)
for i, name in enumerate(BAND_NAMES):              # taller bands drawn on top
    draw(plot[plot['floor_band'] == name], BAND_COLS[i], 3 + i)

ax.set_axis_off()
ax.set_title(f'{AREA_NAME.title()}: floors per building', loc='left')
ax.legend(handles=[Patch(color=c, label=n) for n, c in zip(BAND_NAMES, BAND_COLS)] + [Patch(color=NODATA, label='No height')],
          title='Floors', frameon=False, loc='lower right', fontsize=9)
xmin, ymin, xmax, ymax = plot.total_bounds
bar = 1000 if (xmax - xmin) < 15000 else 5000
ax.plot([xmin, xmin + bar], [ymin, ymin], color=INK, lw=2)
ax.text(xmin, ymin + (ymax - ymin) * 0.01, f'{bar // 1000} km', fontsize=9)
fig.tight_layout(); save(fig, 'v4_map_floors.png'); plt.show()

### V5. Grid maps: density and verticality
Buildings summarised in square cells of `GRID_M` metres. The grid is also saved as a GeoPackage for QGIS.

In [ ]:
pts = res.to_crs(UTM).copy()
pts['gx'] = (pts.centroid.x // GRID_M) * GRID_M
pts['gy'] = (pts.centroid.y // GRID_M) * GRID_M
pts['floor_area_m2'] = pts['footprint_m2'] * pts['floors_final'].fillna(1)
pts['is_tall'] = pts['floors_final'] >= 4

g = pts.groupby(['gx', 'gy']).agg(
        buildings=('bid', 'size'),
        mean_floors=('floors_final', 'mean'),
        max_height=('h_median', 'max'),
        share_4plus=('is_tall', 'mean'),
        floor_area_m2=('floor_area_m2', 'sum'),
        footprint_m2=('footprint_m2', 'sum')).reset_index()
g['far'] = g['floor_area_m2'] / GRID_M ** 2           # built floor area per unit land (floor-area ratio)
g['coverage'] = g['footprint_m2'] / GRID_M ** 2
grid = gpd.GeoDataFrame(g, geometry=[box(x, y, x + GRID_M, y + GRID_M) for x, y in zip(g.gx, g.gy)], crs=UTM)
grid['geometry'] = grid.intersection(aoi.to_crs(UTM).union_all() if hasattr(aoi.geometry, 'union_all') else aoi.to_crs(UTM).unary_union)
MIN_BLDG = 5                                           # averages from fewer buildings are left grey
for c in ['mean_floors', 'share_4plus']:
    grid.loc[grid['buildings'] < MIN_BLDG, c] = np.nan
grid.to_crs(4326).to_file(os.path.join(OUT_DIR, f'grid_{GRID_M}m_{AREA_NAME}.gpkg'), driver='GPKG')

panels = [('buildings', 'Buildings per cell', '{:,.0f}'),
          ('mean_floors', 'Mean floors', '{:.1f}'),
          ('share_4plus', 'Share of buildings with 4+ floors', '{:.0%}'),
          ('far', 'Built floor area ÷ land area', '{:.1f}')]
cmap = mpl.colors.LinearSegmentedColormap.from_list('seq', SEQ)
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (col, title, fmt) in zip(axes.flat, panels):
    vmax = grid[col].quantile(0.98)                     # stop a few extreme cells washing out the map
    grid.plot(column=col, ax=ax, cmap=cmap, vmin=0, vmax=vmax, edgecolor=SURFACE, linewidth=0.3,
              missing_kwds={'color': NODATA, 'edgecolor': SURFACE, 'linewidth': 0.3})
    aoi.to_crs(UTM).boundary.plot(ax=ax, color=INK2, linewidth=0.6)
    ax.set_axis_off(); ax.set_title(title, loc='left')
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.Normalize(0, vmax))
    cb = fig.colorbar(sm, ax=ax, shrink=0.55, pad=0.01)
    cb.outline.set_visible(False); cb.ax.tick_params(labelsize=8, color=MUTED)
    cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _ , f=fmt: f.format(v)))
fig.text(0.01, 0.005, f'Grey: fewer than {MIN_BLDG} buildings in the cell', color=MUTED, fontsize=9)
fig.suptitle(f'{AREA_NAME.title()}: {GRID_M} m grid', x=0.01, ha='left', fontsize=13, fontweight='bold')
fig.tight_layout(); save(fig, 'v5_grid_maps.png'); plt.show()

### V6. Height vs footprint size

In [ ]:
d = res[(res['h_median'] >= MIN_HEIGHT_M) & (res['footprint_m2'] > 0)]
fig, ax = plt.subplots(figsize=(7.5, 5.2))
hb = ax.hexbin(d['footprint_m2'], d['h_median'], xscale='log', gridsize=45, mincnt=1,
               cmap=mpl.colors.LinearSegmentedColormap.from_list('seq', SEQ[1:]), bins='log', linewidths=0.2, edgecolors=SURFACE)
cb = fig.colorbar(hb, ax=ax, pad=0.01); cb.set_label('Buildings per hexagon', color=INK2); cb.outline.set_visible(False)
cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:,.0f}')); cb.ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax.set(title='Height vs footprint size', xlabel='Footprint area (m², log scale)', ylabel='Median height (m)')
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
fig.tight_layout(); save(fig, 'v6_height_vs_footprint.png'); plt.show()

### V7. 3D (2.5D) view of a neighbourhood
Interactive: drag to rotate, scroll to zoom, hover for details. Also saved as an HTML file you can open in a browser.
Set `VIEW_CENTER = (lat, lon)` in Settings to look at a different area.

In [ ]:
import pydeck as pdk

if VIEW_CENTER is None:
    top = res.loc[res['h_median'].idxmax()]
    VIEW_CENTER = (top['lat'], top['lon'])
centre = gpd.GeoSeries(gpd.points_from_xy([VIEW_CENTER[1]], [VIEW_CENTER[0]]), crs=4326).to_crs(UTM)
win = res[res.to_crs(UTM).intersects(centre.buffer(VIEW_RADIUS).iloc[0])].copy()
win = win[win['h_median'] > 0]
print(len(win), 'buildings in 3D view')

def hex2rgb(h): return [int(h[i:i + 2], 16) for i in (1, 3, 5)]
band_rgb = {n: hex2rgb(c) for n, c in zip(BAND_NAMES, BAND_COLS)}
win['fill'] = win['floor_band'].astype(object).map(band_rgb)
win['fill'] = win['fill'].apply(lambda v: v if isinstance(v, list) else hex2rgb(NODATA))
win['label'] = win['name'].replace('', 'Unnamed building')
win['height_txt'] = win['h_median'].round(1)
win['floors_txt'] = win['floors_final'].fillna(-1).astype(int).astype(str).replace('-1', 'n/a')

layer = pdk.Layer('GeoJsonLayer',
                  data=json.loads(win[['label', 'height_txt', 'floors_txt', 'floor_source', 'h_median', 'fill', 'geometry']].to_json()),
                  extruded=True, wireframe=True, pickable=True,
                  get_elevation='properties.h_median', get_fill_color='properties.fill',
                  get_line_color=[255, 255, 255])
deck = pdk.Deck(layers=[layer],
                initial_view_state=pdk.ViewState(latitude=VIEW_CENTER[0], longitude=VIEW_CENTER[1], zoom=15.5, pitch=50, bearing=-20),
                map_style=pdk.map_styles.CARTO_LIGHT,
                tooltip={'text': '{label}\nHeight: {height_txt} m\nFloors: {floors_txt} ({floor_source})'})
html = os.path.join(FIG_DIR, 'v7_3d_view.html')
deck.to_html(html, open_browser=False)
print('saved', html)
deck